# 📚 Projet 12 — Système RAG pour normes BTP (Google Colab)

Pose une question sur les normes de construction → le système **cherche** les bons passages puis **rédige** une réponse sourcée.

**Pipeline :** Corpus → Chunking → Embeddings → Recherche (FAISS) → Génération (LLM gratuit)

**Avant de lancer :** `Exécution` → `Modifier le type d'exécution` → **GPU (T4)**, puis `Tout exécuter`.

## 1. Installer les librairies

In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers gradio
print('✓ Installation terminée')

## 2. Le corpus : normes BTP d'exemple
10 extraits de normes/règles de construction (fictifs mais réalistes).

In [ ]:
corpus = [
    "Échafaudages : un échafaudage de pied peut atteindre une hauteur maximale de 24 mètres sans étude technique spécifique. Au-delà, une note de calcul par un bureau d'études est obligatoire. Les garde-corps doivent être posés à 1 mètre de hauteur.",
    "Équipements de protection individuelle (EPI) : le port du casque est obligatoire sur tout chantier présentant un risque de chute d'objets. Le gilet haute visibilité est requis à proximité d'engins ou de circulation. Les chaussures de sécurité sont obligatoires en permanence.",
    "Travail en hauteur : dès qu'un salarié travaille à plus de 3 mètres du sol, une protection collective (garde-corps, filet) doit être installée en priorité. À défaut, un harnais antichute relié à un point d'ancrage certifié est exigé.",
    "Désamiantage : tout retrait de matériaux contenant de l'amiante doit être réalisé par une entreprise certifiée. Un confinement de la zone et un contrôle d'empoussièrement sont obligatoires. Les déchets amiantés sont évacués en filière agréée.",
    "Béton : le temps de séchage avant décoffrage d'une dalle en béton est d'environ 48 à 72 heures selon la température. La résistance nominale est atteinte à 28 jours. Par temps froid (moins de 5°C), la prise est ralentie et des adjuvants sont recommandés.",
    "Isolation thermique : la résistance thermique R minimale recommandée pour une toiture est de 6 m².K/W en rénovation. Pour les murs, une valeur R de 3,7 m².K/W est visée pour bénéficier des aides à la rénovation énergétique.",
    "Installation électrique : toute installation neuve doit respecter la norme NF C 15-100. Un disjoncteur différentiel 30 mA est obligatoire pour protéger les circuits des pièces d'eau. La mise à la terre de l'installation est impérative.",
    "Bruit sur chantier : au-delà de 85 décibels d'exposition quotidienne, le port de protections auditives est obligatoire. Les travaux bruyants en zone habitée sont interdits avant 7h et après 20h en semaine.",
    "Plomberie : les canalisations d'eau potable doivent être en cuivre, PER ou multicouche. La pression de service ne doit pas dépasser 3 bars ; un réducteur de pression est installé si le réseau dépasse cette valeur.",
    "Charges et planchers : un plancher d'habitation doit supporter une charge d'exploitation minimale de 150 kg/m². Pour un local de stockage ou commercial, cette valeur monte à 250 kg/m² ou plus selon l'usage.",
]
print(f'✓ Corpus : {len(corpus)} documents')
for i, d in enumerate(corpus):
    print(f'  [{i}] {d[:60]}...')

## 3. Embeddings + index FAISS
Chaque document devient un vecteur (son "sens"). FAISS permet de retrouver les plus proches très vite.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, numpy as np

# Modèle d'embeddings multilingue (comprend le français)
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Transformer chaque document en vecteur
embeddings = embedder.encode(corpus, convert_to_numpy=True, normalize_embeddings=True)
print('Forme des embeddings :', embeddings.shape, '(10 docs × 384 dimensions)')

# Construire l'index FAISS (recherche par similarité cosinus)
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print('✓ Index FAISS prêt :', index.ntotal, 'documents indexés')

## 4. La recherche (Retrieval) : trouver les passages pertinents

In [ ]:
def rechercher(question, k=3):
    """Retourne les k documents les plus pertinents pour la question."""
    q_vec = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(q_vec, k)
    return [(corpus[i], float(s)) for i, s in zip(indices[0], scores[0])]

# Test
for doc, score in rechercher("Quelle hauteur maximale pour un échafaudage ?"):
    print(f'[score {score:.2f}] {doc[:70]}...')

## 5. La génération : le LLM rédige la réponse
Modèle gratuit multilingue `bigscience/mt0-large` (aucune clé API).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# On charge le modèle directement (plus robuste que pipeline)
gen_tokenizer = AutoTokenizer.from_pretrained('bigscience/mt0-large')
gen_model = AutoModelForSeq2SeqLM.from_pretrained('bigscience/mt0-large').to(device)

def generer(prompt, max_new_tokens=200):
    """Envoie le prompt au LLM et renvoie le texte généré."""
    inputs = gen_tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    sortie = gen_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return gen_tokenizer.decode(sortie[0], skip_special_tokens=True)

print('✓ Modèle de génération chargé')

## 6. Le RAG complet : chercher PUIS répondre

In [ ]:
def rag(question, k=3):
    # 1. Chercher les passages pertinents
    passages = rechercher(question, k)
    contexte = '\n'.join(f'- {doc}' for doc, _ in passages)

    # 2. Construire le prompt pour le LLM
    prompt = (
        f"Contexte (normes BTP) :\n{contexte}\n\n"
        f"Question : {question}\n"
        f"Réponds en français uniquement à partir du contexte ci-dessus."
    )

    # 3. Générer la réponse
    reponse = generer(prompt)

    # Préparer l'affichage des sources
    sources = '\n'.join(f'• [score {s:.2f}] {doc[:80]}...' for doc, s in passages)
    return reponse, sources

# Test
rep, src = rag("À partir de quelle hauteur faut-il une protection antichute ?")
print('RÉPONSE :', rep)
print('\nSOURCES :\n', src)

## 7. Interface Gradio — pose tes questions 🚀

In [ ]:
import gradio as gr

def repondre(question):
    if not question.strip():
        return '', ''
    return rag(question)

demo = gr.Interface(
    fn=repondre,
    inputs=gr.Textbox(label='❓ Ta question sur les normes BTP', placeholder='Ex : Quelle résistance thermique pour une toiture ?'),
    outputs=[
        gr.Textbox(label='💬 Réponse'),
        gr.Textbox(label='📄 Sources utilisées (passages récupérés)'),
    ],
    title='📚 Assistant Normes BTP (RAG)',
    description='Pose une question ; le système cherche dans les normes puis répond.',
    examples=[
        'Quelle hauteur maximale pour un échafaudage ?',
        'Quand le casque est-il obligatoire ?',
        'Combien de temps pour le séchage du béton ?',
        'Quelle norme pour une installation électrique ?',
    ],
)
demo.launch(share=True)